# 03 — Assignment: RAG Over Every File Format (CSV, JSON, Excel, SQLite)

The crash course leaves an assignment: *"use any of the other file formats — Excel, CSV,
whatever format you want — and complete the same pipeline."* This notebook completes it
for **four additional formats**, then proves the point of the `Document` abstraction:

> Once any data source is converted to LangChain `Document`s, the rest of the pipeline
> (chunking → embedding → vector store → retrieval) needs **zero changes**.

| Format | Loader used | One `Document` per... |
|---|---|---|
| CSV | `CSVLoader` (LangChain built-in) | row |
| JSON | custom loader (stdlib `json`) | FAQ entry |
| Excel `.xlsx` | custom loader (`openpyxl`) | sheet row |
| SQLite `.db` | custom loader (stdlib `sqlite3`) | table row |

The sample files were created by `scripts/generate_sample_data.py`:
`data/csv/ai_models.csv`, `data/json/rag_faq.json`, `data/excel/vector_databases.xlsx`,
and `data/sqlite/llm_catalog.db`.

## 0. Setup

This notebook **reuses the modular `src/` pipeline** — that's the payoff of the modular
refactor: the notebook only demonstrates, the logic lives in importable modules.

In [1]:
import sys

sys.path.append("..")  # make the src/ package importable from notebooks/

from src.data_loader import (
    load_all_documents,
    _load_csv_files,
    _load_excel_files,
    _load_json_files,
    _load_sqlite_files,
)
from pathlib import Path

DATA = Path("../data")
print("Setup complete")

Setup complete


## 1. CSV — `CSVLoader` (LangChain built-in)

`CSVLoader` emits **one `Document` per row**. The `page_content` is formatted as
`column: value` lines — already a natural shape for both the embedding model and the
LLM, so no post-processing is required.

In [2]:
csv_docs = _load_csv_files(DATA)

print()
print("First CSV document:")
print("PAGE CONTENT:")
print(csv_docs[0].page_content)
print()
print("METADATA:", csv_docs[0].metadata)

Found 1 CSV files
  loaded ai_models.csv: 4 rows

First CSV document:
PAGE CONTENT:
model_name: all-MiniLM-L6-v2
model_type: embedding
dimensions: 384
provider: Hugging Face (sentence-transformers)
description: Small open-source sentence embedding model producing 384-dimensional vectors. The default choice for fast RAG prototypes and the model used throughout this pipeline.

METADATA: {'source': '../data/csv/ai_models.csv', 'row': 0, 'source_file': 'ai_models.csv', 'file_type': 'csv'}


## 2. JSON — custom loader (stdlib `json`)

There is no one-size-fits-all way to flatten JSON, so this loader understands our FAQ
shape (`{"faqs": [{question, answer, category}]}`) and emits **one `Document` per
entry**, rendered as `Q: ...\nA: ...` — a user's question then embeds very close to the
stored question text, which is exactly what retrieval needs.

> LangChain's generic `JSONLoader` does this declaratively via a `jq` extraction schema
> (requires the `jq` package). Hand-rolling keeps the mechanics visible: **any**
> structure becomes a `Document` with `page_content` + `metadata`.

In [3]:
json_docs = _load_json_files(DATA)

print()
print("First JSON document:")
print("PAGE CONTENT:")
print(json_docs[0].page_content)
print()
print("METADATA:", json_docs[0].metadata)

Found 1 JSON files
  loaded rag_faq.json: 4 entries

First JSON document:
PAGE CONTENT:
Q: What is chunk overlap and why is it needed?
A: Chunk overlap is the number of characters shared between two consecutive chunks. It preserves continuity so a sentence at a chunk boundary is not cut off from its context. This pipeline uses a 1000-character chunk size with a 200-character overlap.

METADATA: {'source': '../data/json/rag_faq.json', 'source_file': 'rag_faq.json', 'file_type': 'json', 'category': 'chunking', 'entry_index': 0}


## 3. Excel — custom loader (`openpyxl`)

The loader treats the first row of every sheet as the header and emits **one `Document`
per data row**, in the same `column: value` shape as CSVLoader — so downstream behavior
is identical. Sheet name and row number go into the metadata for traceability.

> Ready-made alternative: `UnstructuredExcelLoader` (needs the heavyweight
> `unstructured` package).

In [4]:
excel_docs = _load_excel_files(DATA)

print()
print("First Excel document:")
print("PAGE CONTENT:")
print(excel_docs[0].page_content)
print()
print("METADATA:", excel_docs[0].metadata)

Found 1 Excel files
  loaded vector_databases.xlsx: 4 rows

First Excel document:
PAGE CONTENT:
name: ChromaDB
license: open source (Apache-2.0)
persistence: embedded client, persists to local disk
best_for: prototyping and small-to-medium RAG applications

METADATA: {'source': '../data/excel/vector_databases.xlsx', 'source_file': 'vector_databases.xlsx', 'file_type': 'excel', 'sheet': 'vector_databases', 'row': 2}


## 4. SQL database — custom loader (stdlib `sqlite3`)

The loader discovers every user table through `sqlite_master`, then emits **one
`Document` per row** with the table name in the metadata. The same pattern extends to
any SQL engine (LangChain's `SQLDatabaseLoader` does it through SQLAlchemy).

In [5]:
sqlite_docs = _load_sqlite_files(DATA)

print()
print("First SQLite document:")
print("PAGE CONTENT:")
print(sqlite_docs[0].page_content)
print()
print("METADATA:", sqlite_docs[0].metadata)

Found 1 SQLite databases
  loaded llm_catalog.db: 4 rows from 1 table(s)

First SQLite document:
PAGE CONTENT:
name: llama-3.1-8b-instant
provider: Groq
context_window: 131072
open_weights: yes
notes: Default generation model for this pipeline; very fast inference on Groq hardware.

METADATA: {'source': '../data/sqlite/llm_catalog.db', 'source_file': 'llm_catalog.db', 'file_type': 'sqlite', 'table': 'llm_models', 'row': 1}


## 5. The same pipeline, unchanged, over ALL formats

Now the punchline: run the **exact same** injection pipeline (`FaissVectorStore`
internally does chunk → embed → index → save) over the whole `data/` folder — PDFs,
TXT, CSV, JSON, Excel, and SQLite together — and query it.

Nothing about chunking, embedding, or the vector store knows (or cares) which format a
`Document` came from.

In [6]:
from src.vector_store import FaissVectorStore

# A dedicated persist dir so this notebook doesn't clash with app.py's store.
store = FaissVectorStore(persist_directory="../faiss_store_notebook03")
documents = load_all_documents(str(DATA))
store.build_from_documents(documents)

Loading embedding model for vector store: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Found 3 PDF files
  loaded attention_is_all_you_need.pdf: 3 pages
  loaded embedding_models_report.pdf: 3 pages
  loaded rag_systems_overview.pdf: 2 pages
Found 2 text files
  loaded machine_learning.txt
  loaded python_intro.txt
Found 1 CSV files
  loaded ai_models.csv: 4 rows
Found 1 JSON files
  loaded rag_faq.json: 4 entries
Found 1 Excel files
  loaded vector_databases.xlsx: 4 rows
Found 1 SQLite databases
  loaded llm_catalog.db: 4 rows from 1 table(s)
Total documents loaded: 26
Loading embedding model: all-MiniLM-L6-v2 ...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded. Embedding dimension: 384
Split 26 documents into 26 chunks


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape (26, 384)
Vector store now holds 26 vectors
Saved FAISS index + metadata to '../faiss_store_notebook03'


### Query each format specifically

Each question below was designed to live in exactly one of the new formats — the
retrieval results show the pipeline finding the right file *and* the right row.

In [7]:
queries = [
    ("CSV",    "Which open source embedding model produces 384 dimensional vectors?"),
    ("JSON",   "What is chunk overlap and why is it needed?"),
    ("Excel",  "Which vector database is a fully managed cloud service?"),
    ("SQLite", "Which model has the largest context window?"),
]

for expected_format, query in queries:
    print(f"[expected: {expected_format}] {query}")
    for hit in store.query(query, top_k=2):
        meta = hit["metadata"]
        content_preview = hit["content"][:100].replace("\n", " | ")
        print(f"   {meta['source_file']} ({meta['file_type']}, "
              f"distance {hit['distance']:.3f}): {content_preview}")
    print()

[expected: CSV] Which open source embedding model produces 384 dimensional vectors?
   ai_models.csv (csv, distance 0.628): model_name: text-embedding-3-small | model_type: embedding | dimensions: 1536 | provider: OpenAI | descripti
   ai_models.csv (csv, distance 0.746): model_name: all-MiniLM-L6-v2 | model_type: embedding | dimensions: 384 | provider: Hugging Face (sentence-

[expected: JSON] What is chunk overlap and why is it needed?
   rag_faq.json (json, distance 0.308): Q: What is chunk overlap and why is it needed? | A: Chunk overlap is the number of characters shared b
   rag_systems_overview.pdf (pdf, distance 0.910): Chunking Strategies | Chunking divides documents into smaller pieces that fit within the context size 

[expected: Excel] Which vector database is a fully managed cloud service?
   vector_databases.xlsx (excel, distance 0.780): name: Weaviate | license: open source with a managed cloud offering | persistence: self-hosted or manage
   vector_databases.xlsx (excel

### End-to-end with the LLM (optional — needs `GROQ_API_KEY`)

`RAGSearch` completes the loop: retrieval → augmentation → generation. With `top_k=3`
the LLM sees the neighboring rows too and can compare them (e.g. picking the *largest*
context window out of the retrieved catalog rows).

In [8]:
from src.search import RAGSearch

rag = RAGSearch(store)
result = rag.search_and_summarize(
    "Which model has the largest context window and who provides it?", top_k=3
)
print("ANSWER:")
print(result["answer"][:800])
print()
print("SOURCES:")
for s in result["sources"]:
    print(f"  - {s['source_file']}")

Groq LLM initialized: openai/gpt-oss-120b


ANSWER:
The model with the largest context window is **Claude Sonnet**, and it is provided by **Anthropic**.

SOURCES:
  - llm_catalog.db
  - llm_catalog.db
  - llm_catalog.db


## 6. Wrap-up

Assignment complete — the pipeline now ingests **six formats**:

* **PDF** (`PyPDFLoader`) and **TXT** (`TextLoader`) — from the course itself
* **CSV** (`CSVLoader`), **JSON**, **Excel**, **SQLite** — this assignment

Two takeaways worth internalizing:

1. **The `Document` structure is the contract.** Any source — an S3 bucket, a web page,
   a Postgres table, a Confluence space — joins the pipeline by producing
   `Document(page_content=..., metadata=...)`. Everything downstream is format-blind.
2. **Row-shaped data wants row-shaped Documents.** For tables (CSV/Excel/SQL), one
   Document per row with `column: value` text retrieves far better than dumping the
   whole table into one blob — each row is independently findable, and each stays well
   under the chunk size.

**Next:** `04_transformers_embeddings_deep_dive.ipynb` — how the models underneath all
of this actually work.